# Task 2: Exploratory Data Analysis
Ethiopia Financial Inclusion Forecasting -- Selam Analytics

Figures are written to `../reports/figures/`. This notebook mirrors `src/eda.py`, which can also be run standalone from the project root.

## 1. Dataset composition

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/ethiopia_fi_enriched.csv')
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
obs = df[df.record_type=='observation'].copy()
events = df[df.record_type=='event'].copy()
links = df[df.record_type=='impact_link'].copy()
targets = df[df.record_type=='target'].copy()
df['record_type'].value_counts()

In [ ]:
obs['pillar'].value_counts()

In [ ]:
obs['confidence'].value_counts()

## 2. Temporal coverage
Which years have data for which indicators.

In [ ]:
obs['year'] = obs.observation_date.dt.year
coverage = obs.groupby(['indicator_code','year']).size().unstack(fill_value=0)
coverage

## 3. Access trajectory (2011-2024) & growth rates

In [ ]:
acc = obs[obs.indicator_code=='ACC_OWNERSHIP'].sort_values('observation_date').reset_index(drop=True)
acc['growth_pp'] = acc.value_numeric.diff()
acc['years_elapsed'] = acc.observation_date.dt.year.diff()
acc['pp_per_year'] = acc.growth_pp / acc.years_elapsed
acc[['observation_date','value_numeric','growth_pp','pp_per_year']]

**Key finding:** growth decelerated sharply from +4.5pp/yr (2014-17) to just +1.0pp/yr (2021-24), despite Telebirr's 2021 launch and M-Pesa's 2023 entry -- see Section 6 for the registered-vs-active explanation.

## 4. Gender & urban-rural gaps (2024)

In [ ]:
gap_codes = ['ACC_OWNERSHIP_FEMALE','ACC_OWNERSHIP_MALE','ACC_OWNERSHIP_URBAN','ACC_OWNERSHIP_RURAL']
gap = obs[obs.indicator_code.isin(gap_codes)]
gap[['indicator_code','value_numeric']]

## 5. Usage pillar trend (mobile money & digital payments)

In [ ]:
mm = obs[obs.indicator_code=='ACC_MM_ACCOUNT'].sort_values('observation_date')
dp = obs[obs.indicator_code=='USG_DIGITAL_PAYMENT'].sort_values('observation_date')
mm[['observation_date','value_numeric']]

## 6. Registered vs. active mobile money gap
Telebirr registered users grew from ~27M (2022) to ~54M (2026), while survey-based mobile money account ownership only reached 9.45% of adults (~7-8M) by 2024. This mirrors the Sheet D market-nuance note: many registrations are agent-facilitated, duplicate, or dormant, not unique self-directed active adults.

## 7. Infrastructure / enablers snapshot

In [ ]:
obs[obs.pillar=='enabler'][['indicator','value_numeric','unit']]

## 8. Event timeline & correlation matrix
See `../reports/figures/07_event_timeline.png` and `08_correlation_matrix.png` (produced by `src/eda.py`).

## Key Insights (>=5)
1. **Deceleration despite expansion:** Account ownership growth slowed to +3pp (2021-24) even as Telebirr and M-Pesa combined surpassed 60M registered users -- registration volume is not converting 1:1 into unique adult account ownership.
2. **Access has outpaced Usage historically**, but the two pillars are converging: 49% have an account, yet 35% already make/receive digital payments, and 9.45% specifically via mobile money -- suggesting usage growth (once accounts exist) may now be the more elastic lever.
3. **Large urban-rural and gender gaps persist** (~27pp and ~10pp respectively, 2024 estimates), consistent with Ethiopia's ~78% rural population share limiting agent-network reach.
4. **Infrastructure enablers are still developing**: 4G covers only 33% of the population (2023) even as 3G covers 98% -- a likely constraint on smartphone-based Usage-pillar growth specifically.
5. **P2P has overtaken ATM withdrawals** for the first time (2026), signaling a genuine behavioral shift toward digital-first payment habits, not just registration growth.
6. **Data is sparse**: only 5 Findex Access waves over 13 years and a single Usage-pillar survey point (2024) -- this materially widens forecast uncertainty (see Task 4).